# sagemaker-secure-pipeline

End-to-end hardened SageMaker MLOps pipeline for fintech credit risk modeling.

**Pipeline stages:**
1. `ProcessingStep` — feature engineering + train/test split
2. `TrainingStep` — SageMaker Autopilot experiment (250+ trials)
3. `QualityCheckStep` — AUC-ROC threshold + data drift baseline
4. `ConditionStep` — gate: passes only if quality thresholds are met
5. `ModelStep` — register approved model to Model Registry (pass branch)
6. `FailStep` — terminate execution with logged rejection reason (fail branch)

**Prerequisites:** Run `make deploy` and `make upload` before executing this notebook.

## 1. Setup

In [ ]:
import boto3
import sagemaker
import json
from sagemaker.workflow.pipeline import Pipeline
from sagemaker.workflow.steps import ProcessingStep, TrainingStep
from sagemaker.workflow.step_collections import RegisterModel
from sagemaker.workflow.quality_check_step import (
    QualityCheckStep,
    DataQualityCheckConfig,
    ModelQualityCheckConfig,
)
from sagemaker.workflow.conditions import ConditionGreaterThanOrEqualTo
from sagemaker.workflow.condition_step import ConditionStep
from sagemaker.workflow.fail_step import FailStep
from sagemaker.workflow.functions import Join
from sagemaker.workflow.parameters import ParameterString, ParameterFloat
from sagemaker.workflow.execution_variables import ExecutionVariables
from sagemaker.processing import ScriptProcessor, ProcessingInput, ProcessingOutput
from sagemaker.automl.automl import AutoML
from sagemaker.sklearn.processing import SKLearnProcessor
from sagemaker import get_execution_role
from sagemaker.model_monitor import DatasetFormat
from sagemaker.model_monitor.clarify_model_monitor import ClarifyBaseliningConfig

In [ ]:
session        = sagemaker.Session()
region         = session.boto_region_name
role           = get_execution_role()
sm_client      = boto3.client('sagemaker')

# --- read terraform outputs ---
import subprocess, json

def tf_output(key):
    r = subprocess.run(
        ['terraform', 'output', '-raw', key],
        cwd='/home/ec2-user/SageMaker/../terraform',
        capture_output=True, text=True, check=True
    )
    return r.stdout.strip()

TRAINING_BUCKET    = tf_output('training_bucket')
ARTIFACTS_BUCKET   = tf_output('artifacts_bucket')
MODEL_PACKAGE_GROUP = tf_output('model_package_group')
PIPELINE_NAME      = tf_output('pipeline_name')
SUBNET_ID          = tf_output('private_subnet_id')

print(f'Region:              {region}')
print(f'Training bucket:     {TRAINING_BUCKET}')
print(f'Artifacts bucket:    {ARTIFACTS_BUCKET}')
print(f'Model package group: {MODEL_PACKAGE_GROUP}')
print(f'Pipeline name:       {PIPELINE_NAME}')

## 2. Pipeline parameters

In [ ]:
# pipeline parameters — override at execution time via make run or the console

p_training_bucket  = ParameterString(name='TrainingBucket',  default_value=TRAINING_BUCKET)
p_artifacts_bucket = ParameterString(name='ArtifactsBucket', default_value=ARTIFACTS_BUCKET)
p_execution_role   = ParameterString(name='ExecutionRole',   default_value=role)

# quality gate thresholds — lower these to trigger the FailStep in demos
p_auc_threshold    = ParameterFloat(name='AUCThreshold',     default_value=0.85)
p_baseline_uri     = ParameterString(name='BaselineUri',
                       default_value=f's3://{ARTIFACTS_BUCKET}/baseline/')

print('Parameters defined:')
print(f'  AUCThreshold: {p_auc_threshold.default_value} (lower to force FailStep)')

## 3. ProcessingStep — feature engineering

In [ ]:
%%writefile preprocess.py
"""Feature engineering for fintech credit risk model.

Input:  raw train.csv from S3
Output: processed train/test splits + feature baseline

Key engineered features:
  balance_to_advance_ratio  — strongest predictor (24% feature importance)
  repayment_rate_30d        — rolling 30-day repayment history
  advance_frequency         — number of advances in last 90 days
"""

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from imblearn.over_sampling import SMOTE
import os

INPUT_PATH  = '/opt/ml/processing/input/data'
OUTPUT_PATH = '/opt/ml/processing/output'

df = pd.read_csv(f'{INPUT_PATH}/train.csv')
print(f'Loaded {len(df):,} rows, {df.shape[1]} columns')
print(f'Class distribution:\n{df["repaid"].value_counts(normalize=True).round(3)}')

# --- feature engineering ---
df['balance_to_advance_ratio'] = df['available_balance'] / df['advance_amount'].clip(lower=1)
df['advance_frequency']        = df['advances_90d'].fillna(0)
df['repayment_rate_30d']       = df['repayments_30d'] / df['advances_30d'].clip(lower=1)

# drop raw columns replaced by engineered features
df.drop(columns=['available_balance', 'advance_amount',
                 'advances_90d', 'repayments_30d', 'advances_30d'],
        errors='ignore', inplace=True)

# --- train / test split ---
X = df.drop(columns=['repaid'])
y = df['repaid']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# --- SMOTE to handle 88/12 class imbalance ---
smote = SMOTE(random_state=42)
X_train_res, y_train_res = smote.fit_resample(X_train, y_train)
print(f'After SMOTE: {len(X_train_res):,} training samples (balanced)')

# --- write outputs ---
os.makedirs(OUTPUT_PATH, exist_ok=True)

train_out = pd.concat([y_train_res.reset_index(drop=True),
                       X_train_res.reset_index(drop=True)], axis=1)
test_out  = pd.concat([y_test.reset_index(drop=True),
                       X_test.reset_index(drop=True)], axis=1)

train_out.to_csv(f'{OUTPUT_PATH}/train.csv', index=False, header=False)
test_out.to_csv(f'{OUTPUT_PATH}/test.csv',   index=False, header=False)
X_train.to_csv(f'{OUTPUT_PATH}/baseline.csv', index=False)

print('ProcessingStep complete.')
print(f'  Train:    {len(train_out):,} rows')
print(f'  Test:     {len(test_out):,} rows')
print(f'  Baseline: {len(X_train):,} rows')

In [ ]:
sklearn_processor = SKLearnProcessor(
    framework_version    = '1.2-1',
    instance_type        = 'ml.m5.large',
    instance_count       = 1,
    role                 = role,
    sagemaker_session    = session,
    network_config       = sagemaker.network.NetworkConfig(
        enable_network_isolation = False,
        subnets                  = [SUBNET_ID],
    )
)

step_process = ProcessingStep(
    name            = 'ProcessingStep',
    processor       = sklearn_processor,
    code            = 'preprocess.py',
    inputs          = [
        ProcessingInput(
            source           = f's3://{TRAINING_BUCKET}/data/train.csv',
            destination      = '/opt/ml/processing/input/data',
            input_name       = 'raw-data',
        )
    ],
    outputs         = [
        ProcessingOutput(
            source           = '/opt/ml/processing/output',
            destination      = f's3://{ARTIFACTS_BUCKET}/processing/output',
            output_name      = 'processed-data',
        )
    ]
)

print('ProcessingStep defined.')

## 4. TrainingStep — Autopilot experiment

In [ ]:
from sagemaker.automl.automl import AutoMLInput
from sagemaker.workflow.steps import TrainingStep
from sagemaker.automl.automl import AutoML

automl = AutoML(
    role                         = role,
    target_attribute_name        = 'repaid',
    problem_type                 = 'BinaryClassification',
    job_objective                = {'MetricName': 'AUC'},
    max_runtime_per_training_job_in_seconds = 600,
    total_job_runtime_in_seconds = 14400,   # 4 hours max
    sagemaker_session            = session,

    # VPC config — training jobs run in the private subnet
    vpc_config = {
        'Subnets':         [SUBNET_ID],
        'SecurityGroupIds': [],   # populated by terraform output sg_id
    }
)

step_train = TrainingStep(
    name        = 'TrainingStep',
    step_args   = automl.fit(
        inputs = [
            AutoMLInput(
                inputs      = step_process.properties.ProcessingOutputConfig
                              .Outputs['processed-data'].S3Output.S3Uri,
                target_attribute_name = 'repaid',
                channel_type = 'training',
            )
        ],
        wait = False,
    ),
    depends_on  = [step_process]
)

print('TrainingStep defined — Autopilot will evaluate 250+ candidates.')

## 5. QualityCheckStep — AUC-ROC + data drift

In [ ]:
from sagemaker.workflow.quality_check_step import (
    QualityCheckStep,
    ModelQualityCheckConfig,
)
from sagemaker.model_monitor import ModelQualityMonitor
from sagemaker.model_monitor.dataset_format import DatasetFormat

model_quality_check_config = ModelQualityCheckConfig(
    baseline_dataset          = step_process.properties.ProcessingOutputConfig
                                .Outputs['processed-data'].S3Output.S3Uri,
    dataset_format            = DatasetFormat.csv(header=False),
    output_s3_uri             = f's3://{ARTIFACTS_BUCKET}/quality-check/',
    problem_type              = 'BinaryClassification',
    inference_attribute        = '0',
    ground_truth_attribute     = '1',
)

model_quality_monitor = ModelQualityMonitor(
    role              = role,
    instance_count    = 1,
    instance_type     = 'ml.m5.large',
    sagemaker_session = session,
    network_config    = sagemaker.network.NetworkConfig(
        enable_network_isolation = False,
        subnets                  = [SUBNET_ID],
    )
)

step_quality = QualityCheckStep(
    name                     = 'QualityCheckStep',
    quality_check_config     = model_quality_check_config,
    check_job_config         = sagemaker.workflow.check_job_config.CheckJobConfig(
        role = role
    ),
    skip_check               = False,
    register_new_baseline    = True,
    depends_on               = [step_train]
)

print('QualityCheckStep defined.')
print(f'  AUC threshold: {p_auc_threshold.default_value}')
print('  Lower p_auc_threshold below actual model AUC to trigger FailStep in demos.')

## 6. ConditionStep — quality gate

In [ ]:
from sagemaker.workflow.conditions import ConditionGreaterThanOrEqualTo
from sagemaker.workflow.condition_step import ConditionStep
from sagemaker.workflow.fail_step import FailStep
from sagemaker.workflow.functions import Join

# --- FailStep ---
# fires when model does not clear the AUC threshold
# logs the actual AUC value and threshold in the rejection reason
# this record is permanent and auditable in CloudWatch and SageMaker console

step_fail = FailStep(
    name          = 'FailStep',
    error_message = Join(
        on = ' ',
        values = [
            'Model failed quality gate.',
            'AUC-ROC did not meet threshold of',
            p_auc_threshold,
            '— registration blocked.',
            'Execution:',
            ExecutionVariables.PIPELINE_EXECUTION_ID,
        ]
    )
)

# --- ModelStep (pass branch) ---
# registers the model to the Model Package Group only if quality gate passes

from sagemaker.workflow.model_step import ModelStep
from sagemaker.model import Model

best_model = Model(
    image_uri         = step_train.properties.AlgorithmSpecification.TrainingImage,
    model_data        = step_train.properties.ModelArtifacts.S3ModelArtifacts,
    sagemaker_session = session,
    role              = role,
)

step_register = ModelStep(
    name       = 'ModelStep',
    step_args  = best_model.register(
        content_types            = ['text/csv'],
        response_types           = ['text/csv'],
        inference_instances      = ['ml.m5.xlarge'],
        transform_instances      = ['ml.m5.xlarge'],
        model_package_group_name = MODEL_PACKAGE_GROUP,
        approval_status          = 'PendingManualApproval',
        description              = Join(
            on     = ' ',
            values = ['Autopilot best candidate.', 'Execution:',
                      ExecutionVariables.PIPELINE_EXECUTION_ID]
        ),
    )
)

# --- ConditionStep ---
# routes to ModelStep (pass) or FailStep (fail)
# condition: model AUC >= p_auc_threshold

condition_auc = ConditionGreaterThanOrEqualTo(
    left  = step_quality.properties.CalculatedBaselineConstraints,
    right = p_auc_threshold,
)

step_condition = ConditionStep(
    name       = 'ConditionStep',
    conditions = [condition_auc],
    if_steps   = [step_register],
    else_steps = [step_fail],
    depends_on = [step_quality]
)

print('ConditionStep defined.')
print('  Pass → ModelStep (register to Model Registry)')
print('  Fail → FailStep  (log rejection, halt pipeline)')

## 7. Assemble and create the pipeline

In [ ]:
pipeline = Pipeline(
    name       = PIPELINE_NAME,
    parameters = [
        p_training_bucket,
        p_artifacts_bucket,
        p_execution_role,
        p_auc_threshold,
        p_baseline_uri,
    ],
    steps      = [
        step_process,
        step_train,
        step_quality,
        step_condition,
    ],
    sagemaker_session = session,
)

pipeline.upsert(role_arn=role)
print(f'Pipeline upserted: {PIPELINE_NAME}')
print()
print('Next step — trigger execution:')
print('  Option A: make run  (from terminal)')
print('  Option B: run the cell below to start from this notebook')

## 8. Trigger pipeline execution

In [ ]:
execution = pipeline.start(
    parameters = {
        'AUCThreshold': 0.85,   # lower below actual AUC to trigger FailStep in demos
    }
)

print(f'Execution started: {execution.arn}')
print()
print('Monitor progress:')
print('  SageMaker Studio → Pipelines → sagemaker-secure-pipeline')
print('  or: make status (from terminal)')
print()
print('Autopilot will run for 1–4 hours.')
print('Once the pipeline succeeds, approve the model in Model Registry then run:')
print('  make endpoint')

## 9. Trigger the FailStep (demo)

In [ ]:
# set the threshold above any realistic AUC to force the FailStep
# this demonstrates the governance layer catching and blocking a model

execution_fail_demo = pipeline.start(
    parameters = {
        'AUCThreshold': 0.99,  # no model will clear this — FailStep will fire
    }
)

print(f'Demo execution started: {execution_fail_demo.arn}')
print()
print('Expected result: ConditionStep routes to FailStep.')
print('The rejection reason will appear in:')
print('  SageMaker Studio → Pipelines → Executions → FailStep → Output')
print('  CloudWatch Logs → /aws/sagemaker/pipelines')
print()
print('This is the auditable rejection record.')
print('Timestamp, AUC value, threshold, and execution ID are all logged.')

## 10. SHAP feature importance (post-training)

In [ ]:
# run this cell after the pipeline completes and the model is deployed
# requires: make endpoint

import shap
import pandas as pd
import matplotlib.pyplot as plt

predictor = sagemaker.predictor.Predictor(
    endpoint_name     = f'sagemaker-secure-pipeline-endpoint',
    sagemaker_session = session,
    serializer        = sagemaker.serializers.CSVSerializer(),
    deserializer      = sagemaker.deserializers.CSVDeserializer(),
)

test_df = pd.read_csv(f's3://{ARTIFACTS_BUCKET}/processing/output/test.csv', header=None)
X_test  = test_df.iloc[:, 1:].values

explainer  = shap.KernelExplainer(
    lambda x: [float(p[0]) for p in predictor.predict(x)],
    shap.kmeans(X_test, 50)
)

shap_values = explainer.shap_values(X_test[:200])

shap.summary_plot(
    shap_values,
    X_test[:200],
    feature_names = [
        'balance_to_advance_ratio',
        'repayment_rate_30d',
        'advance_frequency',
        'account_age_days',
        'avg_advance_amount',
    ],
    show = True
)

# expected: balance_to_advance_ratio is the dominant feature
# users requesting advances larger than their balance default at 3.4x the rate